# Predicción del ruido aerodinámico mediante un pipeline de Machine Learning con PySpark

Este proyecto implementa un flujo de ingeniería de datos y aprendizaje automático para predecir el nivel de ruido generado por perfiles aerodinámicos.

El proceso comprende la preparación de datos, construcción de un pipeline de regresión, evaluación del modelo y almacenamiento para su posterior reutilización.

**Tecnologías:** Python, PySpark, Spark MLlib y Docker.

## 1. Preparación de datos (ETL)

### 1.1. Importación de bibliotecas

Se importan las bibliotecas necesarias para trabajar con archivos, procesar información mediante PySpark y construir posteriormente el pipeline de Machine Learning.

In [1]:
import os
import urllib.request
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

### 1.2. Inicialización de Spark

Se crea una sesión local de Apache Spark para realizar las transformaciones de datos y entrenar el modelo de regresión.

In [2]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Airfoil Noise Prediction")
    .getOrCreate()
)

print("Spark:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/01 20:20:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.2.0


### 1.3. Carga del conjunto de datos

Se descarga el conjunto de datos aerodinámicos y se carga en un DataFrame de PySpark para comenzar la exploración y limpieza.

La dirección de descarga se obtiene desde una variable de entorno.

In [3]:
dataset_url = os.getenv("AIRFOIL_DATA_URL")

if not dataset_url:
    raise ValueError("AIRFOIL_DATA_URL no está configurada.")

dataset_path = Path("/tmp/NASA_airfoil_noise_raw.csv")

if not dataset_path.exists():
    urllib.request.urlretrieve(
        dataset_url,
        dataset_path
    )

print("Dataset disponible:", dataset_path.exists())

Dataset disponible: True


In [4]:
df = spark.read.csv(
    dataset_path.as_uri(),
    header=True,
    inferSchema=True
)

### 1.4. Exploración inicial

Se visualizan los primeros cinco registros y el esquema del DataFrame para identificar las columnas disponibles, sus tipos de datos y la variable que se utilizará como objetivo de predicción.

In [5]:
df.show(5)

+---------+-------------+-----------+------------------+-----------------------+----------+
|Frequency|AngleOfAttack|ChordLength|FreeStreamVelocity|SuctionSideDisplacement|SoundLevel|
+---------+-------------+-----------+------------------+-----------------------+----------+
|      800|          0.0|     0.3048|              71.3|             0.00266337|   126.201|
|     1000|          0.0|     0.3048|              71.3|             0.00266337|   125.201|
|     1250|          0.0|     0.3048|              71.3|             0.00266337|   125.951|
|     1600|          0.0|     0.3048|              71.3|             0.00266337|   127.591|
|     2000|          0.0|     0.3048|              71.3|             0.00266337|   127.461|
+---------+-------------+-----------+------------------+-----------------------+----------+
only showing top 5 rows


### 1.5. Exploración de la estructura

Se examina el esquema del conjunto de datos para identificar los tipos de datos y comprobar que las variables predictoras sean numéricas.

In [6]:
df.printSchema()

root
 |-- Frequency: integer (nullable = true)
 |-- AngleOfAttack: double (nullable = true)
 |-- ChordLength: double (nullable = true)
 |-- FreeStreamVelocity: double (nullable = true)
 |-- SuctionSideDisplacement: double (nullable = true)
 |-- SoundLevel: double (nullable = true)



### 1.6. Eliminación de registros duplicados

Se identifican y eliminan los registros duplicados para evitar que observaciones idénticas tengan un peso adicional durante el entrenamiento del modelo.

In [7]:
# Cantidad original de registros
rowcount1 = df.count()

# Eliminar duplicados
df = df.dropDuplicates()

# Cantidad después de la limpieza
rowcount2 = df.count()

print(f"Registros originales: {rowcount1}")
print(f"Registros sin duplicados: {rowcount2}")
print(f"Duplicados eliminados: {rowcount1 - rowcount2}")

Registros originales: 1522
Registros sin duplicados: 1503
Duplicados eliminados: 19


### 1.7. Tratamiento de valores nulos

Se eliminan las filas que contienen valores ausentes para garantizar que los registros utilizados en el entrenamiento tengan información completa.

In [8]:
# Eliminar registros con valores nulos o NaN
df = df.na.drop()

# Cantidad final de registros
rowcount3 = df.count()

print(f"Registros antes: {rowcount2}")
print(f"Registros después: {rowcount3}")
print(f"Registros eliminados: {rowcount2 - rowcount3}")

Registros antes: 1503
Registros después: 1499
Registros eliminados: 4


### 1.8. Renombrado de la variable objetivo

Se renombra `SoundLevel` a `SoundLevelDecibels` para expresar claramente que la variable objetivo representa el nivel de ruido medido en decibelios.

In [9]:
df = df.withColumnRenamed(
    "SoundLevel",
    "SoundLevelDecibels"
)

df.printSchema()

root
 |-- Frequency: integer (nullable = true)
 |-- AngleOfAttack: double (nullable = true)
 |-- ChordLength: double (nullable = true)
 |-- FreeStreamVelocity: double (nullable = true)
 |-- SuctionSideDisplacement: double (nullable = true)
 |-- SoundLevelDecibels: double (nullable = true)



### 1.9. Almacenamiento de los datos procesados

El conjunto de datos limpio se almacena en formato Parquet, que permite conservar el esquema y optimizar el almacenamiento y la lectura de datos para las siguientes etapas.

In [10]:
output_path = Path(
    "/workspace/output/NASA_airfoil_noise_cleaned.parquet"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

df.write \
    .mode("overwrite") \
    .parquet(output_path.as_uri())

print("Archivo Parquet generado correctamente")
print("Ubicación:", output_path)

Archivo Parquet generado correctamente
Ubicación: /workspace/output/NASA_airfoil_noise_cleaned.parquet


In [11]:
print("=== Validación ETL ===")

print("Registros originales:", rowcount1)
print("Registros sin duplicados:", rowcount2)
print("Registros después de eliminar nulos:", rowcount3)
print("Variable objetivo:", df.columns[-1])
print("Parquet generado:", output_path.is_dir())

=== Validación ETL ===
Registros originales: 1522
Registros sin duplicados: 1503
Registros después de eliminar nulos: 1499
Variable objetivo: SoundLevelDecibels
Parquet generado: True


## 2. Construcción del pipeline de Machine Learning

### 2.1. Carga de los datos procesados

Se recupera el conjunto de datos limpio almacenado en formato Parquet para utilizarlo en el entrenamiento del modelo predictivo.

In [12]:
df = spark.read.parquet(output_path.as_uri())

df.show(5)

+---------+-------------+-----------+------------------+-----------------------+------------------+
|Frequency|AngleOfAttack|ChordLength|FreeStreamVelocity|SuctionSideDisplacement|SoundLevelDecibels|
+---------+-------------+-----------+------------------+-----------------------+------------------+
|      400|          0.0|     0.3048|              31.7|             0.00331266|           125.045|
|    16000|          1.5|     0.3048|              71.3|             0.00336729|           106.582|
|      630|          3.0|     0.3048|              55.5|             0.00452492|           129.569|
|      315|          4.0|     0.2286|              31.7|             0.00509068|           123.609|
|      630|          4.0|     0.2286|              31.7|             0.00509068|           130.349|
+---------+-------------+-----------+------------------+-----------------------+------------------+
only showing top 5 rows


In [13]:
rowcount4 = df.count()

print(f"Registros disponibles: {rowcount4}")

Registros disponibles: 1499


### 2.2. Preparación de las variables predictoras

Se seleccionan las variables de entrada que utilizará el modelo para predecir el nivel de ruido aerodinámico.

La variable `SoundLevelDecibels` se excluye porque representa el valor que queremos predecir.

In [14]:
input_cols = [
    col for col in df.columns
    if col != "SoundLevelDecibels"
]

print("Variables predictoras:", input_cols)
print("Variable objetivo: SoundLevelDecibels")

Variables predictoras: ['Frequency', 'AngleOfAttack', 'ChordLength', 'FreeStreamVelocity', 'SuctionSideDisplacement']
Variable objetivo: SoundLevelDecibels


### 2.3. Agrupación de variables con VectorAssembler

Se utiliza `VectorAssembler` para combinar las variables predictoras en una única columna vectorial denominada `features`.

Esta estructura es necesaria para que los algoritmos de Machine Learning de PySpark puedan procesar múltiples variables de entrada.

In [15]:
assembler = VectorAssembler(
    inputCols=input_cols,
    outputCol="features"
)

### 2.4. Escalado de variables con StandardScaler

Se aplica `StandardScaler` para normalizar la escala de las variables predictoras mediante su desviación estándar.

El resultado se almacena en una nueva columna denominada `scaledFeatures`.

In [16]:
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withStd=True,
    withMean=False
)

### 2.5. Configuración del modelo de regresión lineal

Se configura un modelo de regresión lineal para predecir `SoundLevelDecibels` utilizando las variables transformadas en `scaledFeatures`.

El modelo aprenderá la relación entre las características aerodinámicas y el nivel de ruido registrado.

In [17]:
lr = LinearRegression(
    featuresCol="scaledFeatures",
    labelCol="SoundLevelDecibels",
    predictionCol="prediction"
)

### 2.6. Construcción del pipeline

Se integran las tres etapas en un único pipeline de Machine Learning.

Esta estructura permite ejecutar secuencialmente la preparación de variables, el escalado y el entrenamiento del modelo.

In [18]:
pipeline = Pipeline(
    stages=[
        assembler,
        scaler,
        lr
    ]
)

In [19]:
for i, stage in enumerate(pipeline.getStages(), start=1):
    print(f"Etapa {i}: {type(stage).__name__}")

Etapa 1: VectorAssembler
Etapa 2: StandardScaler
Etapa 3: LinearRegression


### 2.7. División del conjunto de datos

Se divide el conjunto de datos en dos grupos:

- **70 % para entrenamiento:** permite que el modelo aprenda las relaciones entre las variables.
- **30 % para prueba:** permite evaluar su desempeño utilizando registros que no participaron en el entrenamiento.

Se utiliza una semilla de 42 para reproducir la división de datos.

In [20]:
trainingData, testingData = df.randomSplit(
    [0.7, 0.3],
    seed=42
)

In [21]:
train_count = trainingData.count()
test_count = testingData.count()

print(f"Registros de entrenamiento: {train_count}")
print(f"Registros de prueba: {test_count}")
print(f"Total: {train_count + test_count}")

Registros de entrenamiento: 1101
Registros de prueba: 398
Total: 1499


### 2.8. Entrenamiento del pipeline

Se entrena el pipeline utilizando exclusivamente el conjunto de entrenamiento.

Durante este proceso, PySpark ajusta el escalador y calcula los parámetros del modelo de regresión lineal.

In [22]:
pipelineModel = pipeline.fit(trainingData)

print("Pipeline entrenado correctamente")

26/11/01 20:37:13 WARN Instrumentation: [4977b6d7] regParam is zero, which might cause numerical instability and overfitting.
netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


Pipeline entrenado correctamente


### 2.9. Validación del pipeline

Se verifica la cantidad de registros procesados, las etapas configuradas y la variable objetivo utilizada por el modelo.

In [23]:
print("=== Validación del pipeline ===")

print("Total de registros:", rowcount4)

for i, stage in enumerate(pipeline.getStages(), start=1):
    print(f"Etapa {i}: {type(stage).__name__}")

print("Variable objetivo:", lr.getLabelCol())
print("Pipeline entrenado:", pipelineModel is not None)

=== Validación del pipeline ===
Total de registros: 1499
Etapa 1: VectorAssembler
Etapa 2: StandardScaler
Etapa 3: LinearRegression
Variable objetivo: SoundLevelDecibels
Pipeline entrenado: True


## 3. Evaluación del modelo

### 3.1. Generación de predicciones

Se utiliza el pipeline entrenado para predecir el nivel de ruido aerodinámico sobre el conjunto de prueba.

Los resultados permiten comparar los valores reales de `SoundLevelDecibels` con los valores estimados por el modelo.

In [24]:
# Generar predicciones utilizando los datos de prueba
predictions = pipelineModel.transform(testingData)

# Comparar los valores reales con las predicciones
predictions.select(
    "SoundLevelDecibels",
    "prediction"
).show(10)

+------------------+------------------+
|SoundLevelDecibels|        prediction|
+------------------+------------------+
|           128.679|122.59722914376778|
|            133.42|127.37968204568838|
|           119.146|130.34077425074506|
|           116.074|131.11016975113537|
|           134.319|127.12627360125096|
|            125.01|127.89456373905155|
|           125.941|131.06220981224084|
|           130.588|125.73739953848445|
|           128.354|121.53249832197925|
|           121.783|124.20059665619313|
+------------------+------------------+
only showing top 10 rows


### 3.2. Error cuadrático medio (MSE)

Se calcula el error cuadrático medio para evaluar la diferencia entre los valores reales y las predicciones.

Esta métrica penaliza especialmente los errores grandes al elevar al cuadrado las diferencias. Un valor más cercano a cero indica un mejor ajuste.

In [25]:
# Evaluar el error cuadrático medio
mse_evaluator = RegressionEvaluator(
    labelCol="SoundLevelDecibels",
    predictionCol="prediction",
    metricName="mse"
)

mse = mse_evaluator.evaluate(predictions)

print(f"Error cuadrático medio (MSE): {mse:.4f}")

Error cuadrático medio (MSE): 24.9977


### 3.3. Error absoluto medio (MAE)

Se calcula el error absoluto medio para conocer cuánto se desvían, en promedio, las predicciones respecto a los niveles reales de ruido.

A diferencia del MSE, esta métrica expresa el error en la misma unidad de la variable objetivo: decibelios.

In [26]:
# Evaluar el error absoluto medio
mae_evaluator = RegressionEvaluator(
    labelCol="SoundLevelDecibels",
    predictionCol="prediction",
    metricName="mae"
)

mae = mae_evaluator.evaluate(predictions)

print(f"Error absoluto medio (MAE): {mae:.4f} dB")

Error absoluto medio (MAE): 3.9137 dB


### 3.4. Coeficiente de determinación (R²)

Se calcula el coeficiente de determinación para evaluar qué proporción de la variabilidad del nivel de ruido puede explicar el modelo.

Un valor cercano a 1 indica un buen ajuste. Un valor de 0 representa un rendimiento equivalente a predecir constantemente la media de los valores reales, mientras que un valor negativo indica un rendimiento inferior a esa referencia.

In [27]:
# Evaluar el coeficiente de determinación
r2_evaluator = RegressionEvaluator(
    labelCol="SoundLevelDecibels",
    predictionCol="prediction",
    metricName="r2"
)

r2 = r2_evaluator.evaluate(predictions)

print(f"Coeficiente de determinación (R²): {r2:.4f}")

Coeficiente de determinación (R²): 0.4960


### 3.5. Resumen de evaluación

Se presentan las métricas obtenidas durante la evaluación y el intercepto del modelo de regresión lineal entrenado.

Estos resultados permiten valorar la precisión de las predicciones y el ajuste general del modelo.

In [28]:
print("=== Evaluación del modelo ===")

print("Mean Squared Error (MSE):", round(mse, 2))
print("Mean Absolute Error (MAE):", round(mae, 2))
print("R Squared (R²):", round(r2, 2))

# Obtener el modelo de regresión entrenado
lrModel = pipelineModel.stages[-1]

print("Intercepto:", round(lrModel.intercept, 2))

=== Evaluación del modelo ===
Mean Squared Error (MSE): 25.0
Mean Absolute Error (MAE): 3.91
R Squared (R²): 0.5
Intercepto: 132.88


### 3.6. Comparación de valores reales y estimados

Se presentan las predicciones junto con sus valores reales y errores absolutos para facilitar la interpretación de los resultados.

In [29]:
prediction_comparison = (
    predictions
    .select(
        "SoundLevelDecibels",
        "prediction"
    )
    .withColumn(
        "absolute_error",
        F.abs(
            F.col("SoundLevelDecibels") -
            F.col("prediction")
        )
    )
)

prediction_comparison.show(10, truncate=False)

+------------------+------------------+------------------+
|SoundLevelDecibels|prediction        |absolute_error    |
+------------------+------------------+------------------+
|128.679           |122.59722914376778|6.081770856232225 |
|133.42            |127.37968204568838|6.040317954311604 |
|119.146           |130.34077425074506|11.194774250745056|
|116.074           |131.11016975113537|15.036169751135375|
|134.319           |127.12627360125096|7.192726398749031 |
|125.01            |127.89456373905155|2.8845637390515435|
|125.941           |131.06220981224084|5.121209812240835 |
|130.588           |125.73739953848445|4.8506004615155405|
|128.354           |121.53249832197925|6.821501678020766 |
|121.783           |124.20059665619313|2.417596656193126 |
+------------------+------------------+------------------+
only showing top 10 rows


## 4. Persistencia y reutilización del modelo

### 4.1. Almacenamiento del pipeline entrenado

Se almacena el pipeline de Machine Learning en disco para permitir su reutilización sin necesidad de repetir el entrenamiento.

El modelo guardado incluye las etapas de preparación de variables, escalado y regresión lineal.

In [30]:
# Ruta para almacenar el modelo
model_path = Path("/workspace/models/Final_Project")

# Crear el directorio de modelos
model_path.parent.mkdir(parents=True, exist_ok=True)

# Guardar el pipeline entrenado
pipelineModel.write() \
    .overwrite() \
    .save(model_path.as_uri())

print("Modelo guardado correctamente")
print("Ubicación:", model_path)

Modelo guardado correctamente
Ubicación: /workspace/models/Final_Project


### 4.2. Recuperación del modelo

Se recupera el pipeline previamente almacenado para comprobar que sus componentes y parámetros pueden reutilizarse sin ejecutar nuevamente el proceso de entrenamiento.

In [31]:
# Cargar el modelo almacenado
loadedPipelineModel = PipelineModel.load(
    model_path.as_uri()
)

print("Modelo cargado correctamente")

Modelo cargado correctamente


### 4.3. Predicciones con el modelo recuperado

Se utiliza el pipeline cargado para generar nuevas predicciones sobre el conjunto de prueba.

Esta verificación permite comprobar que el modelo conserva su capacidad predictiva después de haber sido almacenado y recuperado.

In [32]:
# Generar predicciones con el modelo recuperado
predictions = loadedPipelineModel.transform(
    testingData
)

In [33]:
predictions.select(
    "SoundLevelDecibels",
    "prediction"
).show(5, truncate=False)

+------------------+------------------+
|SoundLevelDecibels|prediction        |
+------------------+------------------+
|128.679           |122.59722914376778|
|133.42            |127.37968204568838|
|119.146           |130.34077425074506|
|116.074           |131.11016975113537|
|134.319           |127.12627360125096|
+------------------+------------------+
only showing top 5 rows


### 4.4. Validación del pipeline recuperado

Se comprueba que el modelo conserva sus etapas de procesamiento y los coeficientes aprendidos durante el entrenamiento.

Los coeficientes permiten conocer la contribución de cada variable dentro del modelo de regresión lineal.

In [34]:
# Obtener el modelo de regresión entrenado
loadedmodel = loadedPipelineModel.stages[-1]

# Número de etapas del pipeline
totalstages = len(loadedPipelineModel.stages)

# Variables predictoras utilizadas
inputcolumns = loadedPipelineModel.stages[0].getInputCols()

print("=== Validación del modelo recuperado ===")
print("Número de etapas:", totalstages)

for variable, coeficiente in zip(
    inputcolumns,
    loadedmodel.coefficients
):
    print(
        f"Coeficiente de {variable}: "
        f"{coeficiente:.4f}"
    )

=== Validación del modelo recuperado ===
Número de etapas: 3
Coeficiente de Frequency: -3.9906
Coeficiente de AngleOfAttack: -2.2881
Coeficiente de ChordLength: -3.3269
Coeficiente de FreeStreamVelocity: 1.4832
Coeficiente de SuctionSideDisplacement: -2.0551


### 4.5. Verificación del rendimiento del modelo recuperado

Se calculan nuevamente las métricas de evaluación para comprobar que el pipeline conserva el rendimiento predictivo después de ser guardado y cargado.

In [35]:
# Evaluar el modelo recuperado
loaded_mse = mse_evaluator.evaluate(predictions)
loaded_mae = mae_evaluator.evaluate(predictions)
loaded_r2 = r2_evaluator.evaluate(predictions)

print("=== Modelo recuperado ===")

print(f"MSE: {loaded_mse:.2f}")
print(f"MAE: {loaded_mae:.2f}")
print(f"R²: {loaded_r2:.2f}")

=== Modelo recuperado ===
MSE: 25.00
MAE: 3.91
R²: 0.50


### 4.6. Finalización del procesamiento

Una vez completadas las transformaciones, el entrenamiento, la evaluación y la validación del modelo almacenado, se finaliza la sesión Spark para liberar los recursos utilizados.

In [36]:
spark.stop()

print("Sesión Spark finalizada correctamente")

Sesión Spark finalizada correctamente


## 5. Conclusiones

Se implementó un pipeline de Machine Learning con PySpark para predecir el nivel de ruido aerodinámico a partir de cinco variables de entrada.

El proceso incluyó la limpieza de datos, transformación de variables, entrenamiento mediante regresión lineal, evaluación y persistencia del modelo.

**Resultados obtenidos:**

- **MSE:** 25.00
- **MAE:** 3.91 dB
- **R²:** 0.50

El modelo explicó aproximadamente el 50 % de la variabilidad del nivel de ruido y presentó un error absoluto promedio de 3.91 dB.

Los resultados evidencian un desempeño moderado, con oportunidades de mejora mediante otros algoritmos, optimización de hiperparámetros y análisis de las relaciones entre variables.

Finalmente, se verificó la persistencia y recuperación del pipeline entrenado, demostrando su capacidad de reutilización sin necesidad de realizar nuevamente el entrenamiento.